# Выбор локации для скважины

Допустим, вы работаете в добывающей компании «ГлавРосГосНефть». Нужно решить, где бурить новую скважину.

Вам предоставлены пробы нефти в трёх регионах: в каждом 10 000 месторождений, где измерили качество нефти и объём её запасов. Постройте модель машинного обучения, которая поможет определить регион, где добыча принесёт наибольшую прибыль. Проанализируйте возможную прибыль и риски техникой *Bootstrap.*

Шаги для выбора локации:

- В избранном регионе ищут месторождения, для каждого определяют значения признаков;
- Строят модель и оценивают объём запасов;
- Выбирают месторождения с самым высокими оценками значений. Количество месторождений зависит от бюджета компании и стоимости разработки одной скважины;
- Прибыль равна суммарной прибыли отобранных месторождений.

# Данные проекта

- Для обучения модели подходит только линейная регрессия (остальные — недостаточно предсказуемые).
- При разведке региона исследуют 500 точек, из которых с помощью машинного обучения выбирают 200 лучших для разработки.
- Бюджет на разработку скважин в регионе — 10 млрд рублей.
- При нынешних ценах один баррель сырья приносит 450 рублей дохода. Доход с каждой единицы продукта составляет 450 тыс. рублей, поскольку объём указан в тысячах баррелей.
- После оценки рисков нужно оставить лишь те регионы, в которых вероятность убытков меньше 2.5%. Среди них выбирают регион с наибольшей средней прибылью.
- Данные синтетические: детали контрактов и характеристики месторождений не разглашаются.

## Загрузка и подготовка данных

На данном этапе импортируем библиотеки. Откроем файлы с данными и изучим их.

Импортируем основные библиотеки.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn as scikit_learn
import warnings
import random
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy import stats as st 

<div class="alert alert-block alert-success">
<b>Успех:</b> Отлично, что все импорты собраны в первой ячейке ноутбука! Если у того, кто будет запускать твой ноутбук будут отсутствовать некоторые библиотеки, то он это увидит сразу, а не в процессе!
</div>

Откроем файлы с данными. Данные геологоразведки трёх регионов находятся в файлах.

In [2]:
geo0 = pd.read_csv('/datasets/geo_data_0.csv')
geo1 = pd.read_csv('/datasets/geo_data_1.csv')
geo2 = pd.read_csv('/datasets/geo_data_2.csv')

Изучим файлы, посмотрим первые 5 строк датафреймов.

In [3]:
geo0.head()

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


In [4]:
geo1.head()

,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


In [5]:
geo2.head()

,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


Во всех датафреймах по 5 столбцов:

- id — уникальный идентификатор скважины;
- f0, f1, f2 — три признака точек (неважно, что они означают, но сами признаки значимы);
- product — объём запасов в скважине (тыс. баррелей).

Рассмотрим общую информацию о данных.

In [6]:
geo0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


In [7]:
geo1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


In [8]:
geo2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


Во всех таблицах по 100000 строк. Пропущенных значений нет. Cтолбцы с id можно удалить, т.к данная информация не потребуется.

In [9]:
geo0 = geo0.drop(['id'], axis=1)

In [10]:
geo1 = geo1.drop(['id'], axis=1)

In [11]:
geo2 = geo2.drop(['id'], axis=1)

Cтолбцы с id удалены.

<div class="alert alert-block alert-success">
<b>Успех:</b> Данные загружены корректно, первичный осмотр проведен. Согласен, что колонка id нам не нужна.
</div>

Подготовим данные для работы с ними. Определим признаки и целевой признак в данных. Разделим данные на обучающую, валидационную и тестовую выборки. Численные признаки масштабируем — приводим к одному масштабу с помощью стандартизации данных. Напишем функцию, чтобы применить ее ко всем датафреймам.

In [12]:
def prepare (region):
    
    features = region.drop('product', axis=1) # определим признаки
    
    target = region['product'] # определим целевой признакак
    
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345) 
    # разделим данные на обучающую, валидационную выборки
    
    numeric = ['f0', 'f1', 'f2'] # выделим численные признаки 
    
    scaler = StandardScaler() # масштабируем численные признаки 
    scaler.fit(features_train[numeric])

    features_train[numeric] = scaler.transform(features_train[numeric])
    features_valid[numeric] = scaler.transform(features_valid[numeric])
    
    return features_train, features_valid, target_train, target_valid 

Вызовем функцию для всех 3 регионов.

In [13]:
features_train0, features_valid0, target_train0, target_valid0 = prepare(geo0)
features_train1, features_valid1, target_train1, target_valid1 = prepare(geo1)
features_train2, features_valid2, target_train2, target_valid2 = prepare(geo2)

# Вывод

На этом этапе подготовили данные для работы с ними. В результате работы:

- Данные изучили.

- Предобработку данных выполнили (пропусков нет, все данные в нужном формате, лишние данные удалены)

- Данные разделили на обучающую и валидационную выборки.

- Данные масштабировали с помощью стандартизации данных.

## Обучение и проверка модели

На данном этапе обучим и проверим модель для каждого региона. Для обучения модели подходит только линейная регрессия (остальные — недостаточно предсказуемые). Напишем функцию для обучения модели линейной регресии, получим предсказания на валидационной выборке и расчитаем средний запас предсказанного сырья и RMSE модели.

In [14]:
def training (features_train, target_train, features_valid, target_valid, region_number):
    
    model = LinearRegression()
    
    model.fit(features_train, target_train)
    
    predictions_valid = pd.Series(model.predict(features_valid))
    
    rmse = mean_squared_error(target_valid, predictions_valid)**0.5
    
    prediction_mean = predictions_valid.mean()
    
    target_mean = target_valid.mean()
    
    print(f'Для региона {region_number} средний запас предсказанного сырья составляет {prediction_mean}, истинный запас {target_mean} а RMSE модели {rmse}')
    return predictions_valid, rmse, prediction_mean


Применим функцию для всех регионов.


In [15]:
predictions_valid0, rmse0, prediction_mean0 = training(features_train0, target_train0, features_valid0, target_valid0, 1)
predictions_valid1, rmse1, prediction_mean1 = training(features_train1, target_train1, features_valid1, target_valid1, 2)
predictions_valid2, rmse2, prediction_mean2 = training(features_train2, target_train2, features_valid2, target_valid2, 3)


Для региона 1 средний запас предсказанного сырья составляет 92.59256778438035, истинный запас 92.07859674082927 а RMSE модели 37.5794217150813
Для региона 2 средний запас предсказанного сырья составляет 68.728546895446, истинный запас 68.72313602435997 а RMSE модели 0.893099286775617
Для региона 3 средний запас предсказанного сырья составляет 94.96504596800489, истинный запас 94.88423280885438 а RMSE модели 40.02970873393434


- Все модели предсказывают средний запас близкий к истинным значениям, максимальная разница между истинным и предсказанным запасом различается всего на несколько десятых. 
- Наилучшими предсказанием обладает модель для региона 2, разница между истинным и предсказанныи запасом различается всего на несколько тысячных, RMSE модели близка к 0. 
- Предсказания моделей для 1 и 3 регионов примерно одинаковые, т.к RMSE приблизительно равны.

# Вывод

На данном этапе обучили и проверили модель для каждого региона:
- обучили модель линейной регрессии
- сделали предсказания на валидационной выборке
- сохранили предсказания и правильные ответы на валидационной выборке
- вычислили средний запас предсказанного сырья и RMSE модели.
- выяснили, что все модели предсказывают средний запас близкий к истинным значениям
- обнаружили, что наилучшими предсказанием обладает модель для региона 2
- увидели, что RMSE и предсказания моделей для 1 и 3 регионов примерно одинаковые




<div class="alert alert-block alert-success">
<b>Успех:</b> Модели обучены корректно. Радует, что ты активно используешь функции. Это помогает избежать дублирования кода, а значит уменьшает вероятность совершить ошибку/опечатку!
</div>

## Подготовка к расчёту прибыли

Подготовимся к расчёту прибыли. Все ключевые значения для расчётов сохраним в отдельных переменных. Вспомним условия задачи:

При разведке региона исследуют 500 точек, из которых с помощью машинного обучения выбирают 200 лучших для разработки.
Бюджет на разработку скважин в регионе — 10 млрд рублей.
При нынешних ценах один баррель сырья приносит 450 рублей дохода. Доход с каждой единицы продукта составляет 450 тыс. рублей, поскольку объём указан в тысячах баррелей.
После оценки рисков нужно оставить лишь те регионы, в которых вероятность убытков меньше 2.5%. 

In [16]:
reseach_boreholes = 500
# cкаважины для разведки
best_boreholes = 200 
# скважины для разработки
budget = 10000
# бюджет на разработку скважин (млн.руб.) 
revenue_product = 0.450000
# доход за тысячу баррелей (млн.руб.) 
probability_loss= 0.025 
# вероятность убытков 
bootstrap_samples = 1000 
# кол-во выборок

Рассчитаем достаточный объём сырья для безубыточной разработки новой скважины.

In [17]:
borehole_budget = budget/best_boreholes
# бюджет на разработку одной скважины
print('бюджет на разработку одной скважины - {:.0f} млн. р.'.format(borehole_budget)) 
product_volume = borehole_budget/revenue_product 
# достаточный объем сырья для безубыточной разработки скважины 
print('достаточный объем сырья для безубыточной разработки скважины - {:.0f} тыс. баррелей.'
      .format(product_volume))


бюджет на разработку одной скважины - 50 млн. р.
достаточный объем сырья для безубыточной разработки скважины - 111 тыс. баррелей.


Достаточный объем сырья для того, чтобы разработка скважины была окупаемой - 111 тыс. баррелей. Средние значения запасов сырья для всех регионов ниже требуемого объема. При разработке скважин, выбранных случайным образом, имеется большая вероятность не окупить бюджет. На следующем этапе работы определим скважины с максимальными значениями предсказаний и произведем расчёт прибыли и рисков.

# Вывод

На этом этапе подготовили данные к расчёту прибыли.
- ключевые значения для расчётов сохранили в отдельных переменных
- рассчитали достаточный объём сырья для безубыточной разработки новой скважины
- выяснили, что средние значения запасов сырья для всех регионов ниже достаточного объема сырья 



<div class="alert alert-block alert-success">
<b>Успех:</b> Точка безубыточности найдена верно, сравнение проведено!
</div>

## Расчёт прибыли по выбранным скважинам и предсказаниям модели

Напишем функцию для расчёта прибыли по выбранным скважинам и предсказаниям модели:

In [18]:
def income_function(target_valid, predictions_valid, best_boreholes, region):
    
    predict_200 = predictions_valid.sort_values(ascending=False)[:best_boreholes] 
    # выбираем 200 скважин с максимальными значениями предсказаний.
    target_200 = target_valid.reset_index(drop=True)[predict_200.index].sum()
    # просуммируем целевое значение объёма сырья, соответствующее этим предсказаниям.
    income = ((target_200.sum() * revenue_product) - budget)/1000 
    # посчитаем прибыль по выбранным для разработки скважинам
    print(f'Прибыль для региона {region} - {income:.3} млрд. рублей, целевое значение объёма сырья - {target_200:.6} тыс.баррелей')

Вызовем функцию для всех регионов и проверим полученные значения.

In [19]:
income_function(target_valid0, predictions_valid0, 200, 1)
income_function(target_valid1, predictions_valid1, 200, 2)
income_function(target_valid2, predictions_valid2, 200, 3)

Прибыль для региона 1 - 3.32 млрд. рублей, целевое значение объёма сырья - 29601.8 тыс.баррелей
Прибыль для региона 2 - 2.42 млрд. рублей, целевое значение объёма сырья - 27589.1 тыс.баррелей
Прибыль для региона 3 - 2.71 млрд. рублей, целевое значение объёма сырья - 28245.2 тыс.баррелей


Прибыль от разработки скважин в регионах составляет от 2.4 до 3.3 млрд. рублей. Наибольшее кол-во сырья и прибыль в 1 регионе, наименьшая прибыль во втором регионе, средняя прибыль в 3 регионе.

Напишем функцию для расчёта прибыли для случайных выбранных скважин:

In [20]:
def profit(target_valid, predictions_valid, best_boreholes): 
    
    predict_boreholes = predictions_valid.sort_values(ascending=False) # отсортируем значения 
    
    target_boreholes = target_valid[predict_boreholes.index][:best_boreholes] # определим верные значения
    
    income = target_boreholes.sum() * revenue_product # посчитаем прибыль
    
    return income

# Вывод

На этом этапе расчитали прибыли по выбранным скважинам и предсказаниям модели.
- выбрали скважины с максимальными значениями предсказаний
- просуммировали целевое значение объёма сырья, соответствующее этим предсказаниям.
- рассчитали прибыль для полученного объёма сырья
- выяснили, что максимальная прибыль в регионе 1 - 3.32 млрд. рублей
- вычислили, что минимальная прибыль в регионе 2 - 2.41 млрд. рублей
- написали функцию для расчёта прибыли для случайных выбранных скважин

<div class="alert alert-block alert-info">
<b>Совет: </b> В этом шаге нужно было просто определить 1 функцию для подсчета прибыли, которая бы использовалась в следующем шаге.
</div>

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> Что-то я не понимаю, что от меня требуется. Нужно чтобы была одна функция а не две? 
</div> 

<div class="alert alert-block alert-info">
<b>Совет (ревью 2): </b> Да, все, что нужно – это одна функция. Другие расчеты в этом шаге не нужны (но ты их можешь оставить для себя).
</div>

## Расчет риска и прибыли

Посчитаем риски и прибыль для каждого региона:

In [28]:
def risk_profit (predictions_valid, target_valid, bootstrap_samples, reseach_boreholes, region):
    
    state = np.random.RandomState(12345) # настраиваем случайный элемент
    
    count = 0 # создаем счетчик

    values = [] # здесь будем хранить все значения для 1000 выборок
    
    for i in range(bootstrap_samples): # создаем 1000 выборок
    
        # сбрасываем индексы и случайно выбираем нужное количество скважин
        target_subsample = target_valid.reset_index(drop=True).sample(reseach_boreholes, replace=True, random_state=state)
        
        # выбираем строки соответствующие отобранным строкам в целевом признаке
        pred_subsample = predictions_valid[target_subsample.index]
        
        # считаем выручку для выбранных строк для нужного количества скважин
        revenue = profit(target_subsample, pred_subsample, best_boreholes) - budget
        
        # добавляем выручку в список выручек всех выборок
        values.append(revenue)
        
        # проверяем не является выборка убыточной
        if revenue < 0:
            count +=1

    values = pd.Series(values)
    
    #посчитаем и напечатаем среднюю выручку
    mean = values.mean()
    print(f'Средняя выручка для региона {region}, {mean:.4} млн рублей')
    
    lower = values.quantile(0.025)
    
    upper = values.quantile(0.975)
    
    print(f'lower:{lower}')
    
    print(f'upper:{upper}')
    # посчитаем и напечатаем 95% доверительный интервал
    
    confidence_interval = st.t.interval(0.95, len(values)-1, values.mean(),values.sem())
    
    print("95%-ый доверительный интервал :", confidence_interval)
    
    pvalue = (values < 0).mean()
    
    if pvalue < probability_loss:
        print("Вероятность убытков равна {:.0%} и является допустимой".format(pvalue))
    else:
        print("Вероятность убытков равна {:.0%} и является недопустимой".format(pvalue))    


In [29]:
risk_profit (predictions_valid0, target_valid0, bootstrap_samples, reseach_boreholes, 1)

Средняя выручка для региона 1, 425.9 млн рублей
lower:-102.09009483793592
upper:947.976353358369
95%-ый доверительный интервал : (408.7322070686989, 443.1448467524865)
Вероятность убытков равна 6% и является недопустимой


In [30]:
risk_profit (predictions_valid1, target_valid1, bootstrap_samples, reseach_boreholes, 2)

Средняя выручка для региона 2, 515.2 млн рублей
lower:68.87322537050254
upper:931.547591257049
95%-ый доверительный интервал : (501.62147617632206, 528.8240707102583)
Вероятность убытков равна 1% и является допустимой


In [31]:
risk_profit (predictions_valid2, target_valid2, bootstrap_samples, reseach_boreholes, 3)

Средняя выручка для региона 3, 435.0 млн рублей
lower:-128.88054732978893
upper:969.7069541802657
95%-ый доверительный интервал : (417.4535520413581, 452.5631735241536)
Вероятность убытков равна 6% и является недопустимой


- только во втором регионе вероятность убытков является допустимой 
- наилучший регион для разработки скважины под номером два
- наибольшая средняя выручка 515 млн.руб.
- наименьшая вероятность убытков равна 1% 
- у 1 и 3 региона вероятность убытков равна 6 % и является недопустимой

Предлагаем выбрать локацию для для скважины во втором регионе, т.к в данном месте наибольшая средняя выручка и наименьшая вероятность убытков.

# Вывод

На данном этапе посчитали риски и прибыль для каждого региона:
- применили технику Bootstrap с 1000 выборок, чтобы найти распределение прибыли
- нашли среднюю прибыль, 95%-й доверительный интервал и риск убытков
- предложили регион №2 для разработки скважин и обосновали выбор

# Итог
В результате проектной работы:
- загрузили и подготовили данные для работы 
- обучили и проверили модель для каждого региона
- подготовили данные к расчёту прибыли 
- расчитали прибыль по выбранным скважинам и предсказаниям модели
- посчитали риски и прибыль для каждого региона
- решили задачу, где бурить новую скважину 

<div class="alert alert-block alert-success">
<b>Успех:</b> Приятно видеть вывод в коцне проекта. С выбором региона согласен.
</div>